# Ejemplo 4 — Sintonía PID mediante Root Locus

En este cuaderno se verifica el diseño analítico de un controlador PID para la planta

$$G(s)=\frac{1}{(s+1)(s+2)(s+10)}.$$

El controlador se expresa en forma factorizada como

$$C_{PID}(s)=K\frac{(s+0.1)(s+z_D)}{s}.$$

Se conserva el cero integral en $s=-0.1$ y se utilizan las especificaciones

$$\zeta=0.5,\qquad T_s\approx1.625\ \text{s}.$$

Estas especificaciones conducen al polo deseado

$$s_d\approx-2.462+4.264j.$$

La condición de ángulo se empleará para verificar la ubicación del segundo cero y la condición de magnitud para calcular la ganancia $K$.

In [ ]:
!pip -q install control

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import control as ct

## Definición de la planta

In [ ]:
s = ct.TransferFunction.s
G = 1 / ((s + 1)*(s + 2)*(s + 10))

G

## Cálculo del polo deseado

A partir de

$$T_s\approx\frac{4}{\sigma},$$

se obtiene

$$\sigma=\frac{4}{T_s}.$$

Para $\zeta=0.5$,

$$\omega_d=\sigma\frac{\sqrt{1-\zeta^2}}{\zeta},$$

y el polo deseado es

$$s_d=-\sigma+j\omega_d.$$

In [ ]:
zeta = 0.5
Ts = 1.625

sigma = 4/Ts
omega_d = sigma*np.sqrt(1-zeta**2)/zeta
sd = -sigma + 1j*omega_d

print(f"sigma   = {sigma:.6f}")
print(f"omega_d = {omega_d:.6f}")
print(f"sd      = {sd.real:.6f} {sd.imag:+.6f}j")

## Condición de ángulo

Antes de incluir el segundo cero, el sistema tiene polos en

$$0,\;-1,\;-2,\;-10,$$

y un cero en

$$-0.1.$$

Se calculan los ángulos aportados por estos polos y por el cero integral. El segundo cero debe aportar el ángulo faltante para satisfacer la condición de Root Locus.

In [ ]:
polos = [0, -1, -2, -10]
ceros = [-0.1]

angulos_polos = [np.degrees(np.angle(sd - p)) for p in polos]
angulos_ceros = [np.degrees(np.angle(sd - z)) for z in ceros]

for p, angulo in zip(polos, angulos_polos):
    print(f"Ángulo desde el polo {p:>4}: {angulo:.4f}°")

for z, angulo in zip(ceros, angulos_ceros):
    print(f"Ángulo desde el cero {z:>4}: {angulo:.4f}°")

angulo_actual = sum(angulos_ceros) - sum(angulos_polos)

# Se selecciona la rama equivalente a -180°
theta_z = -180 - angulo_actual

print(f"\nÁngulo actual = {angulo_actual:.4f}°")
print(f"Ángulo requerido del segundo cero = {theta_z:.4f}°")

## Ubicación del segundo cero

Si el segundo cero está ubicado en $s=-z_D$, su contribución angular cumple

$$\tan\theta_z=\frac{\omega_d}{z_D-\sigma}.$$

Por tanto,

$$z_D=\sigma+\frac{\omega_d}{\tan\theta_z}.$$

In [ ]:
zD = sigma + omega_d/np.tan(np.deg2rad(theta_z))

print(f"z_D = {zD:.6f}")
print(f"El segundo cero está en s = {-zD:.6f}")

## Condición de magnitud

Para

$$L(s)=K\frac{(s+0.1)(s+z_D)}{s(s+1)(s+2)(s+10)},$$

la condición de magnitud conduce a

$$K=\frac{|s_d|\,|s_d+1|\,|s_d+2|\,|s_d+10|}{|s_d+0.1|\,|s_d+z_D|}.$$

In [ ]:
K = abs(sd*(sd + 1)*(sd + 2)*(sd + 10)) / abs((sd + 0.1)*(sd + zD))

print(f"K = {K:.6f}")

## Ganancias equivalentes del controlador PID

Al expandir

$$C_{PID}(s)=K\frac{(s+0.1)(s+z_D)}{s},$$

se obtiene

$$C_{PID}(s)=Ks+K(z_D+0.1)+\frac{0.1Kz_D}{s}.$$

Comparando con

$$C_{PID}(s)=K_Ds+K_P+\frac{K_I}{s},$$

las ganancias equivalentes son

$$K_D=K,$$

$$K_P=K(z_D+0.1),$$

$$K_I=0.1Kz_D.$$

In [ ]:
KD = K
KP = K*(zD + 0.1)
KI = 0.1*K*zD

print(f"K_P = {KP:.6f}")
print(f"K_I = {KI:.6f}")
print(f"K_D = {KD:.6f}")

## Polos de lazo cerrado

Se construye el controlador PID y se verifican los polos del sistema de lazo cerrado.

In [ ]:
C_PID = K*(s + 0.1)*(s + zD)/s
T_PID = ct.feedback(C_PID*G, 1)

polos_PID = ct.poles(T_PID)

print("Polos del sistema de lazo cerrado:")
for polo in polos_PID:
    print(f"{polo.real:.6f} {polo.imag:+.6f}j")

## Root Locus

Se representa el Root Locus del sistema compensado sin incluir la ganancia $K$ y se marcan los polos deseados.

In [ ]:
G_PID = (s + 0.1)*(s + zD) / (s*(s + 1)*(s + 2)*(s + 10))

plt.figure(figsize=(8, 6))
ct.root_locus(G_PID, grid=True)

plt.plot(sd.real, sd.imag, 'rx', markersize=10, markeredgewidth=2,
         label='Polos deseados')
plt.plot(sd.real, -sd.imag, 'rx', markersize=10, markeredgewidth=2)

plt.xlabel('Parte real')
plt.ylabel('Parte imaginaria')
plt.title('Root Locus con controlador PID')
plt.legend()
plt.grid(True)
plt.show()

## Comparación de las respuestas al escalón

Se compara el controlador PID con las sintonías P, PI y PD estudiadas anteriormente.

In [ ]:
# Control P
K_P_only = 43.85
T_P = ct.feedback(K_P_only*G, 1)

# Control PI
K_PI = 43.0387
C_PI = K_PI*(s + 0.1)/s
T_PI = ct.feedback(C_PI*G, 1)

# Control PD
zD_PD = 5.49
K_PD = 32.0
C_PD = K_PD*(s + zD_PD)
T_PD = ct.feedback(C_PD*G, 1)

t = np.linspace(0, 10, 1200)
tP, yP = ct.step_response(T_P, T=t)
tPI, yPI = ct.step_response(T_PI, T=t)
tPD, yPD = ct.step_response(T_PD, T=t)
tPID, yPID = ct.step_response(T_PID, T=t)

plt.figure(figsize=(9, 5))
plt.plot(tP, yP, label='P')
plt.plot(tPI, yPI, label='PI')
plt.plot(tPD, yPD, label='PD')
plt.plot(tPID, yPID, label='PID')
plt.axhline(1, linestyle='--', linewidth=1, label='Referencia')

plt.xlabel('Tiempo [s]')
plt.ylabel('Salida')
plt.title('Comparación de las respuestas al escalón')
plt.grid(True)
plt.legend()
plt.show()

## Error de estado estacionario

Debido al polo en el origen introducido por la acción integral, el sistema es de tipo 1. Para una entrada escalón se espera

$$e_{ss}=0.$$

In [ ]:
valor_final = ct.dcgain(T_PID)
ess = 1 - valor_final

print(f"Valor final de la salida = {valor_final:.6f}")
print(f"Error de estado estacionario = {ess:.6e}")